# Neural network

A feed-forward classifier compares `application_only` and `application_history` features from the shared preparation cache. Both variants use the same applicant splits.

Preprocessing is fitted on training rows only. Validation ROC-AUC selects the model; classification metrics use a fixed 0.5 threshold. Holdout and competition data are not evaluated.


In [1]:
%pip install -q pandas numpy "scikit-learn>=1.3" torch

from pathlib import Path
import sys

REPO_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "scripts" / "project_data.py").is_file()),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError("Notebook working directory must be inside this repository.")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import gc
from time import perf_counter
import numpy as np
import pandas as pd
from IPython.display import display
from scripts.project_data import FEATURE_SETS, load_prepared_data, get_partitions
from scripts.experiment_utils import (
    validation_metrics, split_fingerprint, feature_schema_fingerprint,
    package_versions, save_run_record,
)
from sklearn.compose import ColumnTransformer
from sklearn.impute import MissingIndicator, SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

FEATURE_SET = "application_history"  # Alternative: "application_only"
SPLIT_STRATEGY = "stratified"      # Alternative: "random"
RANDOM_STATE = 42
CLASSIFICATION_THRESHOLD = 0.5
ACTIVE_PARTITIONS = ("train", "validation")

## 1. Feature set and shared splits

The prepared cache contains application-only and application-plus-history features with identical applicant assignments. This notebook selects training and validation rows for `FEATURE_SET`, records the split fingerprint, and releases the full cache bundle.


In [2]:
preparation_started = perf_counter()
prepared_data = load_prepared_data()
run_split_fingerprint = split_fingerprint(
    prepared_data["split_ids"][SPLIT_STRATEGY], prepared_data["labels"],
)
data = get_partitions(
    prepared_data, FEATURE_SET, SPLIT_STRATEGY, partitions=ACTIVE_PARTITIONS,
)
raw_inputs = data["raw_inputs"]
labels = data["labels"]
applicant_ids = data["applicant_ids"]
audit_by_partition = data["audit_by_partition"]
numeric_features = data["numeric_features"]
categorical_features = data["categorical_features"]
run_feature_schema = feature_schema_fingerprint(raw_inputs["train"].columns)
prepared_schema_version = prepared_data["schema_version"]
raw_feature_count = raw_inputs["train"].shape[1]
run_row_counts = {part: len(raw_inputs[part]) for part in ACTIVE_PARTITIONS}
run_positive_rates = {part: float(labels[part].mean()) for part in ACTIVE_PARTITIONS}
del prepared_data, data
gc.collect()
assert set(raw_inputs) == {"train", "validation"}
display(pd.DataFrame([
    {"feature_set": FEATURE_SET, "partition": part, "rows": len(frame), "raw_features": frame.shape[1]}
    for part, frame in raw_inputs.items()
]))
print("Protocol:", SPLIT_STRATEGY)

,feature_set,partition,rows,raw_features
0,application_history,train,215257,303
1,application_history,validation,46127,303


Protocol: stratified


## 2. Neural-network preprocessing

Numeric features use training-median imputation and standardization, with unscaled missingness indicators. Entirely missing training columns are retained and imputed to zero. Categorical features use a missing-value placeholder and a training-only one-hot vocabulary; unseen categories produce an all-zero block.

Prepared inputs are dense `float32` arrays. [Imputation](https://scikit-learn.org/stable/modules/generated/sklearn.impute.SimpleImputer.html) and [one-hot encoding](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OneHotEncoder.html).


In [3]:
def make_nn_preprocessor(numeric_features, categorical_features):
    return ColumnTransformer([
        ("numeric", Pipeline([
            ("imputer", SimpleImputer(strategy="median", keep_empty_features=True)),
            ("scaler", StandardScaler()),
        ]), numeric_features),
        ("numeric_missing", MissingIndicator(
            features="missing-only", error_on_new=False, sparse=False,
        ), numeric_features),
        ("categorical", Pipeline([
            ("imputer", SimpleImputer(
                strategy="constant", fill_value="__MISSING__", keep_empty_features=True,
            )),
            ("encoder", OneHotEncoder(
                handle_unknown="ignore", sparse_output=False, dtype=np.float32,
            )),
        ]), categorical_features),
    ], remainder="drop", sparse_threshold=0.0)


nn_preprocessor = make_nn_preprocessor(numeric_features, categorical_features)
nn_inputs = {
    "train": nn_preprocessor.fit_transform(raw_inputs["train"]).astype(np.float32),
    "validation": nn_preprocessor.transform(raw_inputs["validation"]).astype(np.float32),
}
nn_feature_names = nn_preprocessor.get_feature_names_out()
nn_labels = {part: labels[part].to_numpy(dtype=np.float32) for part in ACTIVE_PARTITIONS}

## 3. Input alignment

`nn_inputs[partition]`, `nn_labels[partition]`, and `applicant_ids[partition]` have the same row order. The fitted preprocessor and feature names stay available; raw frames are released before training.

In [4]:
assert len(nn_feature_names) == len(set(nn_feature_names))
for part in ACTIVE_PARTITIONS:
    inputs = nn_inputs[part]
    assert inputs.shape == (len(applicant_ids[part]), len(nn_feature_names))
    assert inputs.dtype == np.float32 and np.isfinite(inputs).all()
    np.testing.assert_array_equal(raw_inputs[part].index.to_numpy(), applicant_ids[part])
    assert labels[part].index.equals(raw_inputs[part].index)
display(pd.DataFrame([
    {"partition": part, "rows": nn_inputs[part].shape[0], "nn_inputs": nn_inputs[part].shape[1]}
    for part in ACTIVE_PARTITIONS
]))
del raw_inputs, inputs
gc.collect()
preparation_seconds = perf_counter() - preparation_started
print(f"Preparation finished in {preparation_seconds:.1f}s. No training yet.")

,partition,rows,nn_inputs
0,train,215257,653
1,validation,46127,653


Preparation finished in 24.7s. No training yet.


## 4. Model configuration

Two hidden layers (128 and 64 units) use ReLU and dropout 0.2, followed by one output logit. AdamW uses learning rate 0.001 and weight decay 0.0001 with unweighted binary cross-entropy and batch size 1024.

Training is capped at 30 epochs with patience of five epochs on validation ROC-AUC; the best weights are restored. Computation uses four CPU threads and seed 42, with no resampling or threshold tuning. [Loss function](https://docs.pytorch.org/docs/stable/generated/torch.nn.BCEWithLogitsLoss.html).


In [5]:
try:
    import torch
    from torch import nn
    from torch.utils.data import DataLoader, TensorDataset
except ImportError as exc:
    raise ImportError("Run %pip install torch in this kernel, then restart/rerun.") from exc

CONFIGURATION_NAME = "starter_v1"
NN_CONFIG = {
    "hidden_units": [128, 64], "dropout": 0.2, "learning_rate": 0.001,
    "weight_decay": 0.0001, "batch_size": 1024, "evaluation_batch_size": 4096,
    "max_epochs": 30, "patience": 5, "threads": 4, "device": "cpu",
}
print(NN_CONFIG)

{'hidden_units': [128, 64], 'dropout': 0.2, 'learning_rate': 0.001, 'weight_decay': 0.0001, 'batch_size': 1024, 'evaluation_batch_size': 4096, 'max_epochs': 30, 'patience': 5, 'threads': 4, 'device': 'cpu'}


In [6]:
class StarterNetwork(nn.Module):
    def __init__(self, input_width, hidden_units, dropout):
        super().__init__()
        layers = []
        previous_width = input_width
        for width in hidden_units:
            layers.extend([nn.Linear(previous_width, width), nn.ReLU(), nn.Dropout(dropout)])
            previous_width = width
        layers.append(nn.Linear(previous_width, 1))
        self.layers = nn.Sequential(*layers)

    def forward(self, values):
        return self.layers(values).squeeze(-1)  # logits: no sigmoid before BCE loss

def predict_nn(model, inputs, batch_size):
    model.eval()
    result = []
    with torch.inference_mode():
        for start in range(0, len(inputs), batch_size):
            batch = torch.from_numpy(inputs[start:start + batch_size])
            result.append(torch.sigmoid(model(batch)).cpu().numpy())
    return np.concatenate(result)

def fit_starter_nn(inputs, outcomes, config, seed):
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.set_num_threads(config["threads"])
    if config["device"] != "cpu":
        raise ValueError("This CPU starter needs a separately reviewed GPU/device pipeline.")
    model = StarterNetwork(inputs["train"].shape[1], config["hidden_units"], config["dropout"])
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=config["learning_rate"], weight_decay=config["weight_decay"],
    )
    criterion = nn.BCEWithLogitsLoss()
    generator = torch.Generator().manual_seed(seed)
    loader = DataLoader(
        TensorDataset(torch.from_numpy(inputs["train"]), torch.from_numpy(outcomes["train"])),
        batch_size=config["batch_size"], shuffle=True, num_workers=0, generator=generator,
    )
    from sklearn.metrics import roc_auc_score
    best_auc, best_epoch, best_weights, stale_epochs = -np.inf, 0, None, 0
    history = []
    started = perf_counter()
    for epoch in range(1, config["max_epochs"] + 1):
        model.train()
        loss_total = 0.0
        for batch_x, batch_y in loader:
            optimizer.zero_grad(set_to_none=True)
            loss = criterion(model(batch_x), batch_y)
            if not torch.isfinite(loss):
                raise FloatingPointError("Non-finite NN loss; stop and inspect preprocessing.")
            loss.backward()
            optimizer.step()
            loss_total += float(loss.detach()) * len(batch_y)
        probabilities = predict_nn(model, inputs["validation"], config["evaluation_batch_size"])
        auc = float(roc_auc_score(outcomes["validation"], probabilities))
        val_loss = validation_metrics(outcomes["validation"], probabilities)["log_loss"]
        train_loss = loss_total / len(outcomes["train"])
        history.append({
            "epoch": epoch, "train_loss": train_loss, "validation_log_loss": val_loss,
            "validation_roc_auc": auc,
        })
        print(
            f"Epoch {epoch:02d}: train loss={train_loss:.5f}, val loss={val_loss:.5f}, "
            f"val AUC={auc:.5f}, elapsed={perf_counter() - started:.1f}s",
            flush=True,
        )
        if auc > best_auc:
            best_auc, best_epoch = auc, epoch
            best_weights = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}
            stale_epochs = 0
        else:
            stale_epochs += 1
            if stale_epochs >= config["patience"]:
                print(f"Early stopping; restoring epoch {best_epoch}.")
                break
    if best_weights is None:
        raise RuntimeError("No valid NN checkpoint was produced.")
    model.load_state_dict(best_weights)
    model.eval()
    return model, history, best_epoch

## 5. Training

The classifier fits training rows only. Validation ROC-AUC controls early stopping and selects the retained model; labeled holdout and competition rows are excluded.


In [7]:
training_started = perf_counter()
model, training_history, best_iteration = fit_starter_nn(nn_inputs, nn_labels, NN_CONFIG, RANDOM_STATE)
training_seconds = perf_counter() - training_started
prediction_started = perf_counter()
validation_probability = predict_nn(model, nn_inputs["validation"], NN_CONFIG["evaluation_batch_size"])
prediction_seconds = perf_counter() - prediction_started
run_parameters = {**NN_CONFIG, "random_state": RANDOM_STATE, "loss": "BCEWithLogitsLoss", "optimizer": "AdamW"}
model_feature_count = nn_inputs["train"].shape[1]
print(f"Selected epoch: {best_iteration}; training time: {training_seconds:.1f}s")
display(pd.DataFrame(training_history))

Epoch 01: train loss=0.26505, val loss=0.24247, val AUC=0.77111, elapsed=4.9s


Epoch 02: train loss=0.24504, val loss=0.24193, val AUC=0.77295, elapsed=9.8s


Epoch 03: train loss=0.24366, val loss=0.24104, val AUC=0.77660, elapsed=14.7s


Epoch 04: train loss=0.24232, val loss=0.24114, val AUC=0.77658, elapsed=19.7s


Epoch 05: train loss=0.24077, val loss=0.24123, val AUC=0.77509, elapsed=24.6s


Epoch 06: train loss=0.23964, val loss=0.24066, val AUC=0.77611, elapsed=29.2s


Epoch 07: train loss=0.23781, val loss=0.24182, val AUC=0.77396, elapsed=34.2s


Epoch 08: train loss=0.23637, val loss=0.24129, val AUC=0.77567, elapsed=38.8s


Early stopping; restoring epoch 3.
Selected epoch: 3; training time: 43.4s


,epoch,train_loss,validation_log_loss,validation_roc_auc
0,1,0.265053,0.242469,0.771112
1,2,0.245039,0.241935,0.772949
2,3,0.243661,0.241039,0.776597
3,4,0.242320,0.241142,0.776580
4,5,0.240770,0.241226,0.775091
5,6,0.239638,0.240659,0.776107
6,7,0.237808,0.241819,0.773962
7,8,0.236367,0.241293,0.775675


## 6. Validation metrics

ROC-AUC is the primary metric, complemented by average precision, log loss, accuracy, precision, recall, and F1. Classification metrics use a fixed 0.5 threshold. The confusion matrix places actual labels in rows and predicted labels in columns.

A run record in `results/metrics/` stores metrics, configuration, timings, learning history, library versions, and split/feature fingerprints for cross-model comparison.


In [8]:
metrics = validation_metrics(
    labels["validation"], validation_probability, threshold=CLASSIFICATION_THRESHOLD,
)
display(pd.DataFrame([
    {"metric": key, "value": value}
    for key, value in metrics.items() if key not in {"confusion_matrix", "threshold"}
]))
counts = metrics["confusion_matrix"]
display(pd.DataFrame(
    [[counts["tn"], counts["fp"]], [counts["fn"], counts["tp"]]],
    index=pd.Index(["0", "1"], name="Actual"),
    columns=pd.Index(["0", "1"], name="Predicted"),
))
run_record = {
    "model": "nn", "feature_set": FEATURE_SET, "split_strategy": SPLIT_STRATEGY,
    "configuration": CONFIGURATION_NAME, "seed": RANDOM_STATE, "evaluation_partition": "validation",
    "optimizing_metric": "roc_auc", "split_fingerprint": run_split_fingerprint,
    "feature_schema_fingerprint": run_feature_schema, "prepared_schema_version": prepared_schema_version,
    "raw_feature_count": raw_feature_count, "model_feature_count": model_feature_count,
    "rows": run_row_counts, "positive_rates": run_positive_rates, "parameters": run_parameters,
    "selected_iteration": best_iteration, "preparation_seconds": preparation_seconds,
    "training_seconds": training_seconds, "prediction_seconds": prediction_seconds,
    "metrics": metrics, "training_history": training_history, "versions": package_versions("torch"),
    "holdout_evaluated": False, "class_weighting": "none", "threshold_tuned": False,
}
result_path = save_run_record(run_record)
print(f"Saved validation results: {result_path}")
print("Holdout not evaluated. No Kaggle submission created.")

,metric,value
0,roc_auc,0.776597
1,average_precision,0.265943
2,accuracy,0.919440
3,f1,0.009595
4,precision,0.642857
5,recall,0.004834
6,log_loss,0.241039


Predicted,0,1
Actual,,
0,42393,10
1,3706,18


Saved validation results: C:\Users\CRNeu\Documents\Data4401\Data4401-Project2\results\metrics\nn_application_history_stratified_starter_v1_20261006T080721021120Z_70241f3b.json
Holdout not evaluated. No Kaggle submission created.


## 7. Applicant-level validation predictions

The existing validation probabilities are saved to `results/predictions/` as a CSV with matching run metadata. Each row contains applicant ID, actual target, predicted probability and label, error type (TN/FP/FN/TP), gender, age, age group, and run ID.

Metadata links predictions to the metric record, split, configuration, and threshold. These artifacts support downstream error and fairness analysis without refitting. Only validation rows are exported; missing audit categories remain `UNKNOWN` and missing numeric ages remain blank.


In [9]:
from scripts import experiment_utils as _prediction_utils

PREDICTION_EXPORT_DIR = None  # Default: results/predictions/ in this repo.
prediction_csv_path, prediction_metadata_path = _prediction_utils.export_validation_predictions(
    applicant_ids=applicant_ids["validation"],
    labels=labels["validation"],
    probabilities=validation_probability,
    audit=audit_by_partition["validation"],
    result_path=result_path,
    directory=PREDICTION_EXPORT_DIR,
)
print(f"Validation predictions: {prediction_csv_path}")
print(f"Run metadata: {prediction_metadata_path}")
print(f"CSV: {prediction_csv_path.stat().st_size / 1024**2:.2f} MiB; metadata: {prediction_metadata_path.stat().st_size / 1024:.1f} KiB")
print(f"Rows: {len(labels['validation']):,}; feature set: {run_record['feature_set']}")
print("Validation-only predictions exported; no model refitted.")


Validation predictions: C:\Users\CRNeu\Documents\Data4401\Data4401-Project2\results\predictions\nn_application_history_stratified_starter_v1_20261006T080721021120Z_70241f3b.csv
Run metadata: C:\Users\CRNeu\Documents\Data4401\Data4401-Project2\results\predictions\nn_application_history_stratified_starter_v1_20261006T080721021120Z_70241f3b.metadata.json
CSV: 3.68 MiB; metadata: 5.0 KiB
Rows: 46,127; feature set: application_history
Validation-only predictions exported; no model refitted.
